# QA Automation Multi-Agent Workflow

This notebook demonstrates an **assignment-ready QA automation workflow** that turns a story + DOM snippet into framework-specific UI tests.

**User inputs (request payload)**

- `user_id`: who is requesting the run (e.g., `qa_lead`, `qa_engineer`). Included in `final_output` and used for run attribution/tagging.

- `session_id`: unique run/session identifier (e.g., `single-001`). Combined with `user_id` to build a stable `thread_id` for checkpoints + interrupt/resume.

- `story_input`: a mock Jira key like `QA-101` **or** raw user story text.

- `framework`: target test framework (`cypress-ts` or `playwright-ts`).

- `html_dom`: an HTML/DOM snippet **or** a key from `HTML_SAMPLES` (e.g., `login_page`).

**What it does (workflow highlights)**

- **Deterministic tools:** read a user story, extract lightweight acceptance criteria, and generate robust locators (prefers `data-testid`).

- **RAG context:** retrieve relevant example tests from a persistent Chroma store, plus optional **run-memory** (summaries embedded into `qa_run_memory`) to reuse patterns across runs.

- **Long-term user memory:** retrieve per-`user_id` memories (stored in `qa_user_memory`) and feed them into generation as secondary guidance.

- **Orchestration:** LangGraph state machine with a small **fan-out/fan-in** (`retrieve_user_memory` + `retrieve_context` + `precompute_locators` in parallel).

- **Human review:** implemented as a true **interrupt/resume** step (non-blocking by default; can auto-resume from queued decisions for the test cases).

- **Outputs:** generates and validates test code for **Cypress (TS)** or **Playwright (TS)**, with targeted fallbacks if structured LLM output is missing.

- **Checkpoints:** uses SQLite-backed checkpoints at `checkpoints/qa_checkpoints.sqlite` (falls back to `MemorySaver` if SQLite saver is unavailable).

**Agent roles**

- **Story Analyst Agent** reads the story and extracts acceptance criteria.

- **Test Author Agent** generates locator suggestions from HTML and produces framework-specific automated test code.

- **Validation Agent** checks generated artifacts for coherence/usability and can request regeneration when needed.

- **Human Review** approves the output or provides revision feedback before finalization.

**Design goal**

- Keep the workflow **framework-flexible**: the same pipeline can target different UI test frameworks by switching the `framework` input.

**Future work (ideas)**

- Collect framework-specific code-generation best practices and up-to-date patterns from the web (and apply them based on the selected framework).

- Add an **API test case generator** path alongside the UI-focused flow.

- Replace mock data with **real API integrations** (e.g., story lookup / app under test metadata) while keeping deterministic fallbacks.

For a more detailed description, see `README.md`.


## Install Dependencies

In [ ]:
# Clean conflicting packages
%pip uninstall -q -y \
  google-adk \
  opentelemetry-api \
  opentelemetry-sdk \
  opentelemetry-proto \
  opentelemetry-exporter-otlp-proto-http \
  opentelemetry-exporter-otlp-proto-common \
  opentelemetry-exporter-gcp-logging \
  langchain \
  langchain-core \
  langchain-openai \
  langsmith \
  langgraph \
  langgraph-prebuilt

In [ ]:
%pip install -q -U langgraph langgraph-checkpoint-sqlite langchain-core langchain-openai langsmith chromadb beautifulsoup4 lxml tiktoken

In [ ]:
import json
import os
import operator
import re
import sqlite3
import textwrap
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Annotated, Any, Dict, List, TypedDict

import chromadb
from bs4 import BeautifulSoup
from IPython.display import HTML, Image, display
from pydantic import BaseModel, SecretStr

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Checkpointing
from langgraph.checkpoint.memory import MemorySaver

try:
    # Provided by the optional `langgraph-checkpoint-sqlite` package
    from langgraph.checkpoint.sqlite import SqliteSaver
except Exception:
    SqliteSaver = None

# HITL interrupts/resume
try:
    from langgraph.types import Command, interrupt
except Exception:
    Command = None
    interrupt = None

try:
    from langgraph.errors import GraphInterrupt
except Exception:
    GraphInterrupt = None

from langgraph.graph import END, START, StateGraph

try:
    from google.colab import userdata
except Exception:
    userdata = None

print("Imports loaded.")

## Environment & API Keys

- `OPENAI_API_KEY` is required (LLM + embeddings).
- `LANGSMITH_API_KEY` is optional (LangSmith tracing/observability).

In [ ]:
# LangSmith + API key setup
os.environ["LANGSMITH_TRACING"] = "true"

if userdata is not None:
    os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
    os.environ["LANGSMITH_PROJECT"] = "QA Automation Multi-Agent"
    openai_api_key_value = userdata.get("OPENAI_API_KEY")
else:
    os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "")
    os.environ["LANGSMITH_PROJECT"] = "QA Automation Multi-Agent"
    openai_api_key_value = os.getenv("OPENAI_API_KEY", "")

openai_api_key = SecretStr(openai_api_key_value) if openai_api_key_value else None

if not openai_api_key:
    raise ValueError("OPENAI_API_KEY is missing. Add it via Colab Secrets or environment variables.")

os.environ["OPENAI_API_KEY"] = openai_api_key.get_secret_value()
print("Environment configured. LangSmith tracing is enabled.")

## Helper Functions

Reusable helpers for displaying outputs, truncation, and formatting.

In [ ]:
def display_graph(runnable, output_png_path: str = "/content/qa_workflow_graph.png"):
    with open(output_png_path, "wb") as f:
        f.write(runnable.get_graph().draw_mermaid_png())
    display(Image(output_png_path, format="png"))


def extract_last_text(messages: List[BaseMessage]) -> str:
    if not messages:
        return ""
    content = messages[-1].content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, dict) and "text" in part:
                parts.append(part["text"])
            else:
                parts.append(str(part))
        return "\n".join(parts)
    return str(content)


print("Helper functions loaded.")

## Mock Data & HTML Samples

Contains mock Jira stories, HTML snippets, supported frameworks and default payload.

In [ ]:
MOCK_JIRA_STORIES = {
    "QA-101": "As a returning user, I want to log in with email and password so that I can access my dashboard.",
    "QA-102": "As a shopper, I want to add a product to cart so that I can purchase it later.",
    "QA-103": "As an admin, I want to deactivate a user account so that access can be revoked securely.",
    "QA-104": "As a customer, I want to reset my password so that I can recover account access.",
    "QA-105": "As a user, I want to update my profile details so that my account information stays current.",
}

HTML_SAMPLES = {
    "login_page": """
    <main data-testid='login-page'>
      <form data-testid='login-form'>
        <input data-testid='email-input' name='email' type='email' />
        <input data-testid='password-input' name='password' type='password' />
        <button data-testid='login-button' type='submit'>Sign In</button>
      </form>
      <div data-testid='dashboard-link'>Dashboard</div>
    </main>
    """,
    "cart_page": """
    <section>
      <article data-testid='product-card'>
        <h3>Noise Canceling Headphones</h3>
        <button aria-label='Add to cart'>Add to cart</button>
      </article>
      <div id='cart-count'>0</div>
    </section>
    """,
    "admin_page": """
    <div class='users-grid'>
      <table>
        <tr><td>Alice</td><td><button>Deactivate</button></td></tr>
      </table>
      <div role='alert' hidden>User deactivated</div>
    </div>
    """,
}

SUPPORTED_FRAMEWORKS = {
    "cypress-ts": "Cypress with TypeScript",
    "playwright-ts": "Playwright with TypeScript",
}

DEFAULT_REQUEST_PAYLOAD = {
    "story_input": "QA-101",
    "framework": "cypress-ts",
    "html_dom": "login_page",
}

print("Mock Jira stories, HTML DOM samples, and framework options loaded.")

## Chroma Example Corpus

Builds and persists an example test corpus into Chroma for RAG retrieval.

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"
CHROMA_PATH = os.path.join(os.getcwd(), "chroma_store")

CHROMA_CLIENT = None
CHROMA_COLLECTION = None
CHROMA_RUN_MEMORY = None
CHROMA_USER_MEMORY = None

print("Chroma config loaded.")

In [ ]:
EXAMPLE_CORPUS = [
    {
        "id": "cy-login-basic",
        "framework": "cypress-ts",
        "tags": ["login", "happy-path", "data-testid"],
        "story": "As a returning user, I want to log in so I can access my dashboard.",
        "acceptance_criteria": [
            "user can submit valid credentials",
            "successful login redirects to dashboard",
        ],
        "dom_hint": "data-testid=email-input, password-input, login-button",
        "test_code": """describe('Login Flow', () => {
  beforeEach(() => {
    cy.visit('/login');
  });

  it('should allow login with valid credentials', () => {
    cy.get(\"[data-testid='email-input']\").type('user@example.com');
    cy.get(\"[data-testid='password-input']\").type('Password123');
    cy.get(\"[data-testid='login-button']\").click();
    cy.url().should('include', '/dashboard');
  });
});""",
    },
    {
        "id": "cy-login-error",
        "framework": "cypress-ts",
        "tags": ["login", "error", "validation"],
        "story": "As a user, I want to see an error for invalid login.",
        "acceptance_criteria": [
            "invalid credentials show validation message",
        ],
        "dom_hint": "data-testid=login-button, alert-message",
        "test_code": """describe('Login Errors', () => {
  it('should show error on invalid credentials', () => {
    cy.visit('/login');
    cy.get(\"[data-testid='login-button']\").click();
    cy.get(\"[data-testid='alert-message']\").should('be.visible');
  });
});""",
    },
    {
        "id": "cy-cart-add",
        "framework": "cypress-ts",
        "tags": ["cart", "add", "aria-label"],
        "story": "As a shopper, I want to add a product to cart.",
        "acceptance_criteria": [
            "user can add an item to cart",
            "cart counter updates",
        ],
        "dom_hint": "button[aria-label='Add to cart'], #cart-count",
        "test_code": """describe('Cart Add', () => {
  it('should add an item to cart', () => {
    cy.visit('/products');
    cy.get(\"button[aria-label='Add to cart']\").click();
    cy.get('#cart-count').should('contain', '1');
  });
});""",
    },
    {
        "id": "cy-admin-deactivate",
        "framework": "cypress-ts",
        "tags": ["admin", "deactivate", "role"],
        "story": "As an admin, I want to deactivate a user account.",
        "acceptance_criteria": [
            "admin can trigger deactivate action",
            "success feedback appears",
        ],
        "dom_hint": "button:contains('Deactivate'), [role='alert']",
        "test_code": """describe('Admin Deactivate User', () => {
  it('should deactivate user and show success', () => {
    cy.visit('/admin/users');
    cy.contains('button', 'Deactivate').click();
    cy.get(\"[role='alert']\").should('be.visible');
  });
});""",
    },
    {
        "id": "cy-reset-password",
        "framework": "cypress-ts",
        "tags": ["reset", "password", "email"],
        "story": "As a customer, I want to reset my password.",
        "acceptance_criteria": [
            "user can request password reset",
            "confirmation message appears",
        ],
        "dom_hint": "data-testid=email-input, reset-button, confirmation",
        "test_code": """describe('Password Reset', () => {
  it('should request a password reset', () => {
    cy.visit('/reset-password');
    cy.get(\"[data-testid='email-input']\").type('user@example.com');
    cy.get(\"[data-testid='reset-button']\").click();
    cy.get(\"[data-testid='confirmation']\").should('be.visible');
  });
});""",
    },
    {
        "id": "cy-profile-update",
        "framework": "cypress-ts",
        "tags": ["profile", "update"],
        "story": "As a user, I want to update my profile details.",
        "acceptance_criteria": [
            "user can save profile updates",
            "success message appears",
        ],
        "dom_hint": "data-testid=profile-form, save-button, toast",
        "test_code": """describe('Profile Update', () => {
  it('should save profile changes', () => {
    cy.visit('/profile');
    cy.get(\"[data-testid='profile-form']\").within(() => {
      cy.get(\"input[name='firstName']\").clear().type('Alex');
    });
    cy.get(\"[data-testid='save-button']\").click();
    cy.get(\"[data-testid='toast']\").should('contain', 'Saved');
  });
});""",
    },
    {
        "id": "pw-login-basic",
        "framework": "playwright-ts",
        "tags": ["login", "happy-path"],
        "story": "As a returning user, I want to log in so I can see my dashboard.",
        "acceptance_criteria": [
            "user can submit valid credentials",
            "dashboard is visible",
        ],
        "dom_hint": "data-testid=email-input, password-input, login-button",
        "test_code": """import { test, expect } from '@playwright/test';

test.describe('Login Flow', () => {
  test('should allow login with valid credentials', async ({ page }) => {
    await page.goto('/login');
    await page.fill(\"[data-testid='email-input']\", 'user@example.com');
    await page.fill(\"[data-testid='password-input']\", 'Password123');
    await page.click(\"[data-testid='login-button']\");
    await expect(page).toHaveURL(/dashboard/);
  });
});""",
    },
    {
        "id": "pw-cart-add",
        "framework": "playwright-ts",
        "tags": ["cart", "add"],
        "story": "As a shopper, I want to add a product to cart.",
        "acceptance_criteria": [
            "user can add item",
            "cart counter updates",
        ],
        "dom_hint": "button[aria-label='Add to cart'], #cart-count",
        "test_code": """import { test, expect } from '@playwright/test';

test.describe('Cart Add', () => {
  test('should add an item to cart', async ({ page }) => {
    await page.goto('/products');
    await page.click(\"button[aria-label='Add to cart']\");
    await expect(page.locator('#cart-count')).toHaveText('1');
  });
});""",
    },
    {
        "id": "pw-reset-password",
        "framework": "playwright-ts",
        "tags": ["reset", "password"],
        "story": "As a customer, I want to reset my password.",
        "acceptance_criteria": [
            "reset request submits",
            "confirmation visible",
        ],
        "dom_hint": "data-testid=email-input, reset-button, confirmation",
        "test_code": """import { test, expect } from '@playwright/test';

test.describe('Password Reset', () => {
  test('should request a password reset', async ({ page }) => {
    await page.goto('/reset-password');
    await page.fill(\"[data-testid='email-input']\", 'user@example.com');
    await page.click(\"[data-testid='reset-button']\");
    await expect(page.locator(\"[data-testid='confirmation']\")).toBeVisible();
  });
});""",
    },
    {
        "id": "pw-admin-deactivate",
        "framework": "playwright-ts",
        "tags": ["admin", "deactivate"],
        "story": "As an admin, I want to deactivate a user account.",
        "acceptance_criteria": [
            "admin can deactivate user",
            "success feedback appears",
        ],
        "dom_hint": "button:has-text('Deactivate'), [role='alert']",
        "test_code": """import { test, expect } from '@playwright/test';

test.describe('Admin Deactivate User', () => {
  test('should deactivate user and show success', async ({ page }) => {
    await page.goto('/admin/users');
    await page.getByRole('button', { name: 'Deactivate' }).click();
    await expect(page.getByRole('alert')).toBeVisible();
  });
});""",
    },
    {
        "id": "cy-search-filter",
        "framework": "cypress-ts",
        "tags": ["search", "filter", "list"],
        "story": "As a user, I want to filter a list by keyword.",
        "acceptance_criteria": [
            "results update after filter",
        ],
        "dom_hint": "data-testid=search-input, results-list",
        "test_code": """describe('Search Filter', () => {
  it('should filter results by keyword', () => {
    cy.visit('/search');
    cy.get(\"[data-testid='search-input']\").type('headphones');
    cy.get(\"[data-testid='results-list']\").should('contain', 'headphones');
  });
});""",
    },
    {
        "id": "pw-toast-success",
        "framework": "playwright-ts",
        "tags": ["toast", "success", "update"],
        "story": "As a user, I want a success toast after saving.",
        "acceptance_criteria": [
            "save action shows success toast",
        ],
        "dom_hint": "data-testid=save-button, toast",
        "test_code": """import { test, expect } from '@playwright/test';

test.describe('Save Feedback', () => {
  test('should show success toast after save', async ({ page }) => {
    await page.goto('/settings');
    await page.click(\"[data-testid='save-button']\");
    await expect(page.locator(\"[data-testid='toast']\")).toContainText('Saved');
  });
});""",
    },
]

print(f"Loaded EXAMPLE_CORPUS with {len(EXAMPLE_CORPUS)} examples.")

In [ ]:
def _build_example_document(example: Dict[str, Any]) -> str:
    criteria_text = "\n".join(f"- {item}" for item in example["acceptance_criteria"])
    return (
        f"Framework: {example['framework']}\n"
        f"Story: {example['story']}\n"
        f"Acceptance Criteria:\n{criteria_text}\n"
        f"DOM Hint: {example['dom_hint']}\n"
        f"Test Code:\n{example['test_code']}"
    )


def _get_chroma_client() -> chromadb.Client:
    global CHROMA_CLIENT

    if CHROMA_CLIENT is None:
        CHROMA_CLIENT = chromadb.PersistentClient(path=CHROMA_PATH)

    return CHROMA_CLIENT


def _ensure_chroma_index() -> None:
    global CHROMA_COLLECTION

    if CHROMA_COLLECTION is not None:
        return

    client = _get_chroma_client()
    CHROMA_COLLECTION = client.get_or_create_collection("qa_example_tests")

    if CHROMA_COLLECTION.count() > 0:
        return

    embedder = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        api_key=openai_api_key.get_secret_value(),
    )

    documents = [_build_example_document(example) for example in EXAMPLE_CORPUS]
    embeddings = embedder.embed_documents(documents)
    ids = [example["id"] for example in EXAMPLE_CORPUS]
    metadatas = [
        {
            "framework": example["framework"],
            "tags": ",".join(example["tags"]),
        }
        for example in EXAMPLE_CORPUS
    ]

    CHROMA_COLLECTION.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )


def _ensure_run_memory_collection() -> None:
    global CHROMA_RUN_MEMORY

    if CHROMA_RUN_MEMORY is not None:
        return

    client = _get_chroma_client()
    CHROMA_RUN_MEMORY = client.get_or_create_collection("qa_run_memory")


def _ensure_user_memory_collection() -> None:
    """Long-term memory keyed by user_id (role isolation)."""
    global CHROMA_USER_MEMORY

    if CHROMA_USER_MEMORY is not None:
        return

    client = _get_chroma_client()
    CHROMA_USER_MEMORY = client.get_or_create_collection("qa_user_memory")


def _summarize_preference_from_feedback(feedback: str) -> str:
    """Deterministic mini-summary: keep it short, stable, and demo-friendly."""
    cleaned = " ".join(feedback.strip().split())
    cleaned = cleaned[:400]
    if not cleaned:
        return ""
    return f"User preference: {cleaned}"


def persist_user_feedback_memory(
    *,
    user_id: str,
    framework: str,
    session_id: str,
    feedback: str,
) -> List[str]:
    """Persist HITL feedback to user-scoped long-term memory."""
    if not feedback or not str(feedback).strip():
        return []

    _ensure_user_memory_collection()

    embedder = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        api_key=openai_api_key.get_secret_value(),
    )

    feedback_text = str(feedback).strip()
    pref_text = _summarize_preference_from_feedback(feedback_text)

    memory_ids: List[str] = []
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
    base = _hash_text(f"{user_id}|{framework}|{feedback_text}")

    docs: List[str] = [feedback_text]
    metas: List[Dict[str, Any]] = [
        {
            "user_id": user_id,
            "framework": framework,
            "session_id": session_id,
            "type": "hitl_feedback",
            "timestamp": timestamp,
        }
    ]
    ids: List[str] = [f"ufb-{base}"]

    if pref_text:
        docs.append(pref_text)
        metas.append(
            {
                "user_id": user_id,
                "framework": framework,
                "session_id": session_id,
                "type": "preference",
                "timestamp": timestamp,
            }
        )
        ids.append(f"upref-{base}")

    embeddings = embedder.embed_documents(docs)

    for doc_id, doc, emb, meta in zip(ids, docs, embeddings, metas):
        try:
            CHROMA_USER_MEMORY.add(
                ids=[doc_id],
                documents=[doc],
                embeddings=[emb],
                metadatas=[meta],
            )
            memory_ids.append(doc_id)
        except Exception:
            continue

    return memory_ids


def retrieve_user_memory_context(*, user_id: str, query: str, k: int = 3) -> Dict[str, Any]:
    """Retrieve top-k user-scoped memory snippets for prompt conditioning."""
    _ensure_user_memory_collection()

    embedder = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        api_key=openai_api_key.get_secret_value(),
    )
    query_embedding = embedder.embed_query(query)

    results = CHROMA_USER_MEMORY.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where={"user_id": user_id},
    )

    ids = results.get("ids", [[]])[0]
    documents = results.get("documents", [[]])[0]

    snippets = [_truncate_text(doc, max_chars=350) for doc in documents if doc]
    context = "User memory: none."
    if snippets:
        context = "User/role preferences & prior feedback:\n- " + "\n- ".join(snippets)

    return {"context": context, "ids": [i for i in ids if i]}


print("Chroma helpers loaded.")

## Tool Definitions

Deterministic tools exposed to agents (reading stories, extracting criteria, generating locators, building templates, retrieval).

In [ ]:
def _slugify(value: str) -> str:
    slug = re.sub(r"[^a-zA-Z0-9]+", "-", value.strip().lower())
    slug = re.sub(r"-+", "-", slug).strip("-")
    return slug or "generated-test"


def _truncate_text(text: str, max_chars: int = 600) -> str:
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 20] + "\n...<truncated>"


def _hash_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:12]


print("Utility helpers loaded.")

In [ ]:
@tool
def read_user_story(story_input: str) -> str:
    """Read a user story from a Jira-like key or return raw story text."""
    if story_input in MOCK_JIRA_STORIES:
        return MOCK_JIRA_STORIES[story_input]
    return story_input


@tool
def extract_acceptance_criteria(user_story: str) -> str:
    """Create lightweight acceptance criteria from a user story."""
    lowered = user_story.lower()
    criteria: List[str] = []

    if "login" in lowered:
        criteria.extend(
            [
                "user can submit valid credentials",
                "successful login redirects to dashboard",
                "failed login shows visible validation message",
            ]
        )
    if "cart" in lowered or "add" in lowered:
        criteria.extend(
            [
                "user can add an item to cart",
                "cart counter is updated after add",
            ]
        )
    if "deactivate" in lowered:
        criteria.extend(
            [
                "admin can trigger deactivate action",
                "success feedback is shown after deactivation",
            ]
        )
    if "reset password" in lowered or "recover" in lowered:
        criteria.extend(
            [
                "user can request a password reset",
                "confirmation message appears after request",
            ]
        )

    if not criteria:
        criteria.extend(
            [
                "primary happy path succeeds",
                "error path provides user-facing feedback",
            ]
        )

    return json.dumps({"criteria": criteria}, indent=2)


print("Tools registered: read_user_story, extract_acceptance_criteria")

In [ ]:
@tool
def generate_locators_from_dom(html_dom: str) -> str:
    """Generate robust locators from an HTML DOM tree, prioritizing data-testid."""
    soup = BeautifulSoup(html_dom, "lxml")
    selectors: Dict[str, str] = {}

    for element in soup.select("[data-testid]"):
        test_id = element.get("data-testid")
        if test_id:
            selectors[test_id] = f"[data-testid='{test_id}']"

    for element in soup.select("[aria-label]"):
        aria = element.get("aria-label")
        key = _slugify(f"aria-{aria}")
        selectors.setdefault(key, f"[aria-label='{aria}']")

    for element in soup.select("[role]"):
        role = element.get("role")
        key = _slugify(f"role-{role}")
        selectors.setdefault(key, f"[role='{role}']")

    for element in soup.select("[id]"):
        node_id = element.get("id")
        key = _slugify(f"id-{node_id}")
        selectors.setdefault(key, f"#{node_id}")

    buttons = soup.find_all("button")
    for idx, button in enumerate(buttons, 1):
        text = (button.get_text() or "").strip()
        if text:
            key = _slugify(f"button-{text}")
            selectors.setdefault(key, f"button:has-text('{text}')")
        else:
            selectors.setdefault(f"button-{idx}", "button")

    if not selectors:
        selectors["page-root"] = "body"

    return json.dumps({"locators": selectors}, indent=2)


print("Tool registered: generate_locators_from_dom")

In [ ]:
@tool
def build_test_template(
    framework: str,
    feature_name: str,
    acceptance_criteria_json: str,
    locators_json: str,
) -> str:
    """Build framework-specific UI automation test code from criteria and locator maps."""
    criteria_payload = json.loads(acceptance_criteria_json)
    locator_payload = json.loads(locators_json)

    criteria = criteria_payload.get("criteria", [])[:3]
    if not criteria:
        criteria = ["primary user flow works as expected"]

    locators = locator_payload.get("locators", {})
    locator_values = list(locators.values())
    primary_selector = locator_values[0] if locator_values else "body"

    safe_feature = feature_name.strip() or "Generated QA Feature"

    if framework == "playwright-ts":
        tests = []
        for item in criteria:
            tests.append(
                """
  test('{criterion}', async ({{ page }}) => {{
    await page.goto('/');
    await expect(page.locator(\"{selector}\")).toBeVisible();
  }});
""".strip().format(criterion=item, selector=primary_selector)
            )

        return "\n".join(
            [
                "import { test, expect } from '@playwright/test';",
                "",
                f"test.describe('{safe_feature}', () => {{",
                textwrap.indent("\n\n".join(tests), "  "),
                "});",
            ]
        )

    tests = []
    for item in criteria:
        tests.append(
            """
  it('{criterion}', () => {{
    cy.visit('/');
    cy.get(\"{selector}\").should('be.visible');
  }});
""".strip().format(criterion=item, selector=primary_selector)
        )

    return "\n".join(
        [
            f"describe('{safe_feature}', () => {{",
            textwrap.indent("\n\n".join(tests), "  "),
            "});",
        ]
    )


print("Tool registered: build_test_template")

In [ ]:
@tool
def retrieve_examples(query: str, framework: str = "", k: int = 3) -> str:
    """Retrieve relevant example tests from Chroma."""
    _ensure_chroma_index()

    embedder = OpenAIEmbeddings(
        model=EMBEDDING_MODEL,
        api_key=openai_api_key.get_secret_value(),
    )
    query_embedding = embedder.embed_query(query)

    where_filter = {"framework": framework} if framework else None
    results = CHROMA_COLLECTION.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=where_filter,
    )

    examples: List[Dict[str, Any]] = []
    ids = results.get("ids", [[]])[0]
    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]

    for idx, example_id in enumerate(ids):
        doc = documents[idx] if idx < len(documents) else ""
        meta = metadatas[idx] if idx < len(metadatas) else {}
        examples.append(
            {
                "id": example_id,
                "framework": meta.get("framework", ""),
                "tags": meta.get("tags", ""),
                "snippet": _truncate_text(doc),
            }
        )

    return json.dumps(
        {
            "query": query,
            "count": len(examples),
            "examples": examples,
        },
        indent=2,
    )


print("Tool registered: retrieve_examples")

## Agent Prompts & Models

System prompts and structured response models used by the Analyst, Generator, and Validator agents.

In [ ]:
ANALYST_SYSTEM_PROMPT = """
You are the Story Analyst Agent for QA automation.

Responsibilities:
- Always call read_user_story first.
- Always call extract_acceptance_criteria second.
- Return concise structured analysis.

Output format requirement:
Return valid JSON with keys:
- story_text
- acceptance_criteria_json
- analysis_summary
""".strip()

GENERATOR_SYSTEM_PROMPT = """
You are the Test Author Agent for QA automation.

Responsibilities:
- Use retrieved examples as primary guidance for structure, selectors, and style.
- Prefer data-testid selectors, avoid hard waits, and ensure assertions exist.
- Call generate_locators_from_dom only if selectors are missing.
- Call retrieve_examples only if retrieval context is missing.
- Produce framework-specific test code from acceptance criteria and locators.

Output format requirement:
Return valid JSON with keys:
- locators_json
- generated_test
- generation_summary
""".strip()

VALIDATOR_SYSTEM_PROMPT = """
You are the Validation Agent for QA automation outputs.

Responsibilities:
- Validate that generated outputs are coherent and usable.
- Flag missing or inconsistent data.
- Reject hard waits (cy.wait(<number>)) or fragile selectors.
- Recommend regeneration only when issues are material.

Output format requirement:
Return valid JSON with keys:
- validation_errors (array of strings)
- validation_summary (string)
- should_regenerate (boolean)
""".strip()


class AnalystOutput(BaseModel):
    story_text: str
    acceptance_criteria_json: str
    analysis_summary: str


class GeneratorOutput(BaseModel):
    locators_json: str
    generated_test: str
    generation_summary: str


class ValidatorOutput(BaseModel):
    validation_errors: list[str]
    validation_summary: str
    should_regenerate: bool


print("Agent prompts and structured output schemas loaded.")

In [ ]:
def _env_str(name: str, default: str) -> str:
    value = os.getenv(name)
    if value is None:
        return default
    value = str(value).strip()
    return value if value else default


def _env_float(name: str, default: float) -> float:
    value = os.getenv(name)
    if value is None:
        return default
    try:
        return float(value)
    except Exception:
        return default


DEFAULT_LLM_MODEL = _env_str("QA_LLM_MODEL", "gpt-4o-mini")
ANALYST_LLM_MODEL = _env_str("QA_ANALYST_MODEL", DEFAULT_LLM_MODEL)
GENERATOR_LLM_MODEL = _env_str("QA_GENERATOR_MODEL", "gpt-4o")
VALIDATOR_LLM_MODEL = _env_str("QA_VALIDATOR_MODEL", DEFAULT_LLM_MODEL)

QA_REASONING_EFFORT = _env_str("QA_REASONING_EFFORT", "low")
QA_TEMPERATURE = _env_float("QA_TEMPERATURE", 0.0)

print(
    "LLM models: "
    f"analyst={ANALYST_LLM_MODEL}, "
    f"generator={GENERATOR_LLM_MODEL}, "
    f"validator={VALIDATOR_LLM_MODEL}"
)

base_model = ChatOpenAI(
    model=DEFAULT_LLM_MODEL,
    api_key=openai_api_key.get_secret_value(),
    reasoning_effort=QA_REASONING_EFFORT,
    temperature=QA_TEMPERATURE,
    response_format={"type": "json_object"},
)

analyst_structured_model = ChatOpenAI(
    model=ANALYST_LLM_MODEL,
    api_key=openai_api_key.get_secret_value(),
    reasoning_effort=QA_REASONING_EFFORT,
    temperature=QA_TEMPERATURE,
    response_format=AnalystOutput,
)

generator_structured_model = ChatOpenAI(
    model=GENERATOR_LLM_MODEL,
    api_key=openai_api_key.get_secret_value(),
    reasoning_effort=QA_REASONING_EFFORT,
    temperature=QA_TEMPERATURE,
    response_format=GeneratorOutput,
)

validator_structured_model = ChatOpenAI(
    model=VALIDATOR_LLM_MODEL,
    api_key=openai_api_key.get_secret_value(),
    reasoning_effort=QA_REASONING_EFFORT,
    temperature=QA_TEMPERATURE,
    response_format=ValidatorOutput,
)

# Bind tools to the model for use in agents
analyst_tools = [read_user_story, extract_acceptance_criteria]
generator_tools = [generate_locators_from_dom, build_test_template, retrieve_examples]

analyst_model = analyst_structured_model.bind_tools(analyst_tools)
generator_model = generator_structured_model.bind_tools(generator_tools)
validator_model = validator_structured_model

analyst_fallback_model = base_model.bind_tools(analyst_tools)
generator_fallback_model = base_model.bind_tools(generator_tools)
validator_fallback_model = base_model

print("Models with tool bindings ready: Analyst and Generator")

## State Schema & Parser

Workflow state schema and request parsing helpers.

In [ ]:
class WorkflowState(TypedDict):
    user_request: str
    user_id: str
    session_id: str
    story_input: str
    framework: str
    html_dom: str
    story_text: str
    acceptance_criteria_json: str
    locators_json: str
    precomputed_locators_json: str
    retrieval_context: str
    retrieval_example_ids: List[str]
    run_memory_context: str
    user_memory_context: str
    user_memory_ids: List[str]
    analysis_chunks: Annotated[List[str], operator.add]
    analysis: str
    generated_test: str
    approved: bool
    hitl_feedback: Annotated[List[str], operator.add]
    hitl_payload: Dict[str, Any]
    validation_errors: List[str]
    validation_summary: str
    run_summary: str
    final_output: str


def parse_user_request_text(user_request: str) -> Dict[str, str]:
    payload: Dict[str, Any]

    try:
        payload = json.loads(user_request)
    except Exception:
        payload = {"story_input": user_request}

    user_id = str(payload.get("user_id", "anonymous")).strip() or "anonymous"
    session_id = str(payload.get("session_id", "")).strip()

    story_input = str(payload.get("story_input") or payload.get("jira_key") or user_request)

    framework = str(payload.get("framework", "cypress-ts")).lower().strip()
    if framework not in SUPPORTED_FRAMEWORKS:
        framework = "cypress-ts"

    html_raw = payload.get("html_dom", "login_page")
    if isinstance(html_raw, str) and html_raw in HTML_SAMPLES:
        html_dom = HTML_SAMPLES[html_raw]
    else:
        html_dom = str(html_raw)

    return {
        "user_id": user_id,
        "session_id": session_id,
        "story_input": story_input,
        "framework": framework,
        "html_dom": html_dom,
    }


def parse_request_node(state: WorkflowState) -> Dict[str, Any]:
    parsed = parse_user_request_text(state["user_request"])
    return {
        **parsed,
        "approved": False,
        "validation_errors": [],
        "validation_summary": "",
        "retrieval_context": "",
        "retrieval_example_ids": [],
        "precomputed_locators_json": "",
        "run_memory_context": "",
        "user_memory_context": "",
        "user_memory_ids": [],
        "analysis_chunks": [],
        "run_summary": "",
        "hitl_payload": {},
    }


print("State schema and request parser ready.")

## Agent Loop (Tool-enabled)

Implementation of `run_agent_loop` supporting tool calls and fallback behavior.

In [ ]:
def run_agent_loop(
    model_with_tools,
    system_prompt: str,
    user_prompt: str,
    max_iterations: int = 5,
    tools: List[Any] = None,
    invoke_config: Dict[str, Any] = None,
    fallback_model: Any = None,
 ) -> str:
    """
    Run an agent loop. If `tools` is provided, uses a LangGraph ToolNode mini-graph
    (agent -> tools -> agent) until the model stops emitting tool calls.

    This replaces the previous manual tool-dispatch implementation and yields better
    trace/debug visibility (distinct agent/tool steps in LangSmith).
    """
    invoke_config = invoke_config or {}
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]

    def _invoke_model(target_messages: List[BaseMessage]) -> AIMessage:
        try:
            return (
                model_with_tools.invoke(target_messages, config=invoke_config)
                if invoke_config
                else model_with_tools.invoke(target_messages)
            )
        except Exception as e:
            if fallback_model is None:
                raise
            print(f"[Agent invoke fallback] structured output failed: {e}")
            return (
                fallback_model.invoke(target_messages, config=invoke_config)
                if invoke_config
                else fallback_model.invoke(target_messages)
            )

    # If no tools, do a single model call (validator path).
    if not tools:
        try:
            response = _invoke_model(messages)
            return extract_last_text([response])
        except Exception as e:
            print(f"[Agent invoke error] {e}")
            return ""

    # Tool-enabled path via ToolNode
    try:
        try:
            from langgraph.prebuilt import ToolNode
        except Exception:
            from langgraph.prebuilt.tool_node import ToolNode  # type: ignore
    except Exception as e:
        print(f"[ToolNode unavailable; falling back to manual loop] {e}")
        # Minimal manual fallback using provided tools list
        tools_by_name = {getattr(t, "name", ""): t for t in tools}
        for iteration in range(max_iterations):
            try:
                response = _invoke_model(messages)
            except Exception as ex:
                print(f"[Agent loop error at iteration {iteration}] {ex}")
                return ""

            if hasattr(response, "tool_calls") and response.tool_calls:
                messages.append(response)
                for tool_call in response.tool_calls:
                    tool_name = tool_call.get("name") or tool_call.get("type")
                    tool_input = tool_call.get("args", {})
                    tool_obj = tools_by_name.get(tool_name)
                    if tool_obj is None:
                        tool_result = f"Unknown tool: {tool_name}"
                    else:
                        try:
                            tool_result = tool_obj.invoke(tool_input)
                        except Exception as ex:
                            tool_result = f"Tool error: {ex}"
                    messages.append(
                        ToolMessage(
                            content=tool_result,
                            tool_call_id=tool_call.get("id", str(iteration)),
                        )
                    )
            else:
                return extract_last_text([response])

        return extract_last_text(messages)

    class AgentMessagesState(TypedDict):
        messages: Annotated[List[BaseMessage], operator.add]

    model_invoke_config = dict(invoke_config)
    graph_invoke_config = dict(invoke_config)
    graph_invoke_config["recursion_limit"] = max(10, 2 * max_iterations + 4)

    tool_node = ToolNode(tools)

    def agent_step(state: AgentMessagesState) -> Dict[str, Any]:
        try:
            response = _invoke_model(state["messages"] )
        except Exception as e:
            print(f"[ToolNode agent_step error] {e}")
            response = AIMessage(content="")
        return {"messages": [response]}

    def route_after_agent(state: AgentMessagesState) -> str:
        if not state.get("messages"):
            return "end"
        last = state["messages"][-1]
        return "tools" if getattr(last, "tool_calls", None) else "end"

    builder = StateGraph(AgentMessagesState)
    builder.add_node("agent", agent_step)
    builder.add_node("tools", tool_node)
    builder.add_node("end", lambda state: {})

    builder.add_edge(START, "agent")
    builder.add_conditional_edges("agent", route_after_agent, ["tools", "end"] )
    builder.add_edge("tools", "agent")
    builder.add_edge("end", END)

    runnable = builder.compile()

    try:
        result = runnable.invoke({"messages": messages}, config=graph_invoke_config)
        return extract_last_text(result.get("messages", []))
    except Exception as e:
        print(f"[ToolNode graph invoke error] {e}")
        return ""

In [ ]:
def retrieve_run_memory_node(state: WorkflowState) -> Dict[str, Any]:
    query = (
        "Story:\n"
        f"{state['story_input']}\n\n"
        "Framework:\n"
        f"{state['framework']}\n\n"
        "DOM Snippet:\n"
        f"{state['html_dom'][:400]}"
    )

    run_memory_context = "Prior run notes: none."
    try:
        _ensure_run_memory_collection()
        embedder = OpenAIEmbeddings(
            model=EMBEDDING_MODEL,
            api_key=openai_api_key.get_secret_value(),
        )
        query_embedding = embedder.embed_query(query)
        where_filter = {"framework": state["framework"]} if state.get("framework") else None
        results = CHROMA_RUN_MEMORY.query(
            query_embeddings=[query_embedding],
            n_results=3,
            where=where_filter,
        )
        documents = results.get("documents", [[]])[0]
        snippets = [_truncate_text(doc, max_chars=400) for doc in documents if doc]
        if snippets:
            run_memory_context = "Prior run notes:\n- " + "\n- ".join(snippets)
    except Exception:
        run_memory_context = "Prior run notes unavailable."

    return {"run_memory_context": run_memory_context}


print("Graph node ready: retrieve_run_memory_node")

In [ ]:
def analyst_node(state: WorkflowState) -> Dict[str, Any]:
    analyst_prompt = f"""
Analyze this QA request and prepare structured criteria.

User request:
{state['user_request']}

Story input:
{state['story_input']}

Prior run notes:
{state.get('run_memory_context', 'None')}

Call read_user_story first, then extract_acceptance_criteria.
Return JSON with story_text, acceptance_criteria_json, and analysis_summary.
""".strip()

    invoke_config = {
        "run_name": "analyst",
        "tags": ["agent", "analyst", state.get("framework", "")],
        "metadata": {
            "user_id": state.get("user_id", ""),
            "session_id": state.get("session_id", ""),
            "framework": state.get("framework", ""),
            "story_input": state.get("story_input", ""),
        },
    }

    analyst_response = run_agent_loop(
        analyst_model,
        ANALYST_SYSTEM_PROMPT,
        analyst_prompt,
        max_iterations=5,
        tools=analyst_tools,
        invoke_config=invoke_config,
        fallback_model=analyst_fallback_model,
    )

    story_text = read_user_story.invoke({"story_input": state["story_input"]})
    acceptance_criteria_json = extract_acceptance_criteria.invoke({"user_story": story_text})
    analysis_summary = analyst_response if analyst_response.strip() else "Story analyzed and criteria extracted."

    try:
        parsed = json.loads(analyst_response)
        story_text = parsed.get("story_text", story_text)
        acceptance_criteria_json = parsed.get("acceptance_criteria_json", acceptance_criteria_json)
        analysis_summary = parsed.get("analysis_summary", analysis_summary)

        if isinstance(acceptance_criteria_json, dict):
            acceptance_criteria_json = json.dumps(acceptance_criteria_json)
    except (json.JSONDecodeError, ValueError):
        pass

    analysis_chunks = [analysis_summary] if analysis_summary else []
    return {
        "story_text": story_text,
        "acceptance_criteria_json": acceptance_criteria_json,
        "analysis": analysis_summary,
        "analysis_chunks": analysis_chunks,
    }


print("Graph node ready: analyst_node")

In [ ]:
def retrieve_user_memory_node(state: WorkflowState) -> Dict[str, Any]:
    """Retrieve long-term, per-user (role) memory to steer generation."""
    user_id = str(state.get("user_id", "")).strip() or "anonymous"

    query = (
        "User:",
        f"{user_id}",
        "\nFramework:",
        f"{state.get('framework', '')}",
        "\nStory:",
        f"{state.get('story_text', '')}",
        "\nAcceptance Criteria:",
        f"{state.get('acceptance_criteria_json', '')}",
    )

    query_text = "\n".join([part for part in query if part])

    try:
        mem = retrieve_user_memory_context(user_id=user_id, query=query_text, k=3)
        ids = mem.get("ids", [])
        context = mem.get("context", "User memory: none.")
        summary = f"Retrieved {len(ids)} user memories." if ids else "Retrieved 0 user memories."
        return {
            "user_memory_context": context,
            "user_memory_ids": ids,
            "analysis_chunks": [summary],
        }
    except Exception:
        return {
            "user_memory_context": "User memory unavailable.",
            "user_memory_ids": [],
            "analysis_chunks": ["User memory retrieval unavailable."],
        }


def retrieve_context_node(state: WorkflowState) -> Dict[str, Any]:
    query = (
        "Story:\n"
        f"{state['story_text']}\n\n"
        "Acceptance Criteria:\n"
        f"{state['acceptance_criteria_json']}\n\n"
        "Framework:\n"
        f"{state['framework']}\n\n"
        "DOM Snippet:\n"
        f"{state['html_dom'][:400]}"
    )

    retrieval_json = retrieve_examples.invoke(
        {"query": query, "framework": state["framework"], "k": 3},
    )

    example_ids: List[str] = []
    retrieval_summary = "Retrieved 0 examples."

    try:
        parsed = json.loads(retrieval_json)
        examples = parsed.get("examples", [])
        example_ids = [ex.get("id", "") for ex in examples if ex.get("id")]
        retrieval_summary = f"Retrieved {len(example_ids)} examples: {', '.join(example_ids)}"
    except (json.JSONDecodeError, ValueError):
        retrieval_json = json.dumps(
            {
                "query": query,
                "count": 0,
                "examples": [],
                "error": "retrieval output was not valid JSON",
            },
            indent=2,
        )

    return {
        "retrieval_context": retrieval_json,
        "retrieval_example_ids": example_ids,
        "analysis_chunks": [retrieval_summary],
    }


def precompute_locators_node(state: WorkflowState) -> Dict[str, Any]:
    precomputed = ""
    summary = "Precomputed 0 locators."

    try:
        precomputed = generate_locators_from_dom.invoke({"html_dom": state["html_dom"]})
        parsed = json.loads(precomputed)
        locators = parsed.get("locators", {}) if isinstance(parsed, dict) else {}
        if isinstance(locators, dict):
            summary = f"Precomputed {len(locators)} locators."
    except Exception:
        precomputed = ""
        summary = "Precompute locators unavailable."

    return {
        "precomputed_locators_json": precomputed,
        "analysis_chunks": [summary],
    }


def prep_join_node(state: WorkflowState) -> Dict[str, Any]:
    chunks = [
        str(chunk).strip()
        for chunk in (state.get("analysis_chunks", []) or [])
        if str(chunk).strip()
    ]

    base = str(state.get("analysis", "")).strip()

    if chunks:
        joined = "\n".join(chunks)
        merged = f"{base}\n\n{joined}".strip() if base else joined
    else:
        merged = base

    return {"analysis": merged}


print(
    "Graph nodes ready: retrieve_user_memory_node, retrieve_context_node, "
    "precompute_locators_node, prep_join_node"
)

In [ ]:
def generator_node(state: WorkflowState) -> Dict[str, Any]:
    accumulated_feedback = "\n".join(state.get("hitl_feedback", []))

    generator_prompt = f"""
Generate framework-specific automation tests.

Framework: {state['framework']}
Story text: {state['story_text']}
Acceptance criteria JSON: {state['acceptance_criteria_json']}

Prior run notes:
{state.get('run_memory_context', '')}

User/role long-term memory:
{state.get('user_memory_context', '')}

Retrieved examples (from Chroma):
{state.get('retrieval_context', '')}

HTML DOM:
{state['html_dom']}

Revision notes from human review:
{accumulated_feedback if accumulated_feedback else 'None'}

Use retrieved examples as the primary style guide.
Use user/role memory as secondary guidance (preferences, recurring feedback).
Call generate_locators_from_dom only if selectors are missing.
Return JSON with locators_json, generated_test, and generation_summary.
""".strip()

    invoke_config = {
        "run_name": "generator",
        "tags": ["agent", "generator", state.get("framework", "")],
        "metadata": {
            "user_id": state.get("user_id", ""),
            "session_id": state.get("session_id", ""),
            "framework": state.get("framework", ""),
            "story_input": state.get("story_input", ""),
        },
    }

    generator_response = run_agent_loop(
        generator_model,
        GENERATOR_SYSTEM_PROMPT,
        generator_prompt,
        max_iterations=5,
        tools=generator_tools,
        invoke_config=invoke_config,
        fallback_model=generator_fallback_model,
    )

    locators_json = ""
    generated_test = ""
    generation_summary = ""

    try:
        parsed = json.loads(generator_response)
        locators_json = parsed.get("locators_json", "")
        generated_test = parsed.get("generated_test", "")
        generation_summary = parsed.get("generation_summary", "")

        if isinstance(locators_json, dict):
            locators_json = json.dumps(locators_json)
    except (json.JSONDecodeError, ValueError):
        pass

    if not locators_json:
        precomputed = state.get("precomputed_locators_json", "")
        if precomputed:
            locators_json = precomputed
        else:
            locators_json = generate_locators_from_dom.invoke(
                {"html_dom": state["html_dom"]}
            )

    if not generated_test:
        feature_name = state["story_text"][:70]
        generated_test = build_test_template.invoke(
            {
                "framework": state["framework"],
                "feature_name": feature_name,
                "acceptance_criteria_json": state["acceptance_criteria_json"],
                "locators_json": locators_json,
            }
        )
        if not generation_summary:
            generation_summary = "Fallback template used due to missing LLM output."

    if not generation_summary:
        generation_summary = "Test generated successfully."

    merged_analysis = (
        state.get("analysis", "")
        + "\n\n"
        + f"Generation summary: {generation_summary}"
    )

    return {
        "locators_json": locators_json,
        "generated_test": generated_test,
        "analysis": merged_analysis,
    }


def validator_node(state: WorkflowState) -> Dict[str, Any]:
    validator_prompt = f"""
Validate the generated QA automation artifacts.

User story: {state['story_text']}
Acceptance criteria JSON: {state['acceptance_criteria_json']}
Locators JSON: {state['locators_json']}
Generated test code:
{state['generated_test']}

HTML DOM:
{state['html_dom']}

Return JSON with validation_errors, validation_summary, and should_regenerate.
""".strip()

    invoke_config = {
        "run_name": "validator",
        "tags": ["agent", "validator", state.get("framework", "")],
        "metadata": {
            "user_id": state.get("user_id", ""),
            "session_id": state.get("session_id", ""),
            "framework": state.get("framework", ""),
            "story_input": state.get("story_input", ""),
        },
    }

    validator_response = run_agent_loop(
        validator_model,
        VALIDATOR_SYSTEM_PROMPT,
        validator_prompt,
        max_iterations=1,
        tools=None,
        invoke_config=invoke_config,
        fallback_model=validator_fallback_model,
    )

    validation_errors: List[str] = []
    validation_summary = "Validation completed."
    should_regenerate = False

    try:
        parsed = json.loads(validator_response)
        validation_errors = parsed.get("validation_errors", []) or []
        validation_summary = parsed.get("validation_summary", validation_summary)
        should_regenerate = bool(parsed.get("should_regenerate", False))

        if isinstance(validation_errors, str):
            validation_errors = [validation_errors]
    except (json.JSONDecodeError, ValueError):
        validation_summary = "Validation response was not JSON; skipped validation."

    if should_regenerate and not validation_errors:
        validation_errors = ["Validator requested regeneration without specific errors."]

    updates: Dict[str, Any] = {
        "validation_errors": validation_errors,
        "validation_summary": validation_summary,
    }

    if validation_errors:
        feedback = "Validation issues:\n- " + "\n- ".join(validation_errors)
        updates["hitl_feedback"] = [feedback]

    return updates


def route_after_validation(state: WorkflowState) -> str:
    return "generator" if state.get("validation_errors") else "human_review"


print("Graph nodes ready: generator_node, validator_node")

In [ ]:
HITL_TEST_DECISIONS: List[Any] = []


def enqueue_hitl_responses(*responses: Any):
    for response in responses:
        HITL_TEST_DECISIONS.append(response)


def _next_hitl_decision(*, allow_input: bool = False) -> Any:
    """Pop the next queued HITL decision, optionally falling back to interactive input."""
    if HITL_TEST_DECISIONS:
        response = HITL_TEST_DECISIONS.pop(0)
        print(f"[Simulated HITL response] {response}")
        return response

    if not allow_input:
        return None

    user_input = input("Human review: type 'approve' or provide revision feedback: ").strip()
    if user_input.lower() in {"approve", "approved", "yes", "y", "true"}:
        return {"decision": "approve"}

    return {"decision": "revise", "feedback": user_input}


print("HITL helpers ready: enqueue_hitl_responses, _next_hitl_decision")

In [ ]:
def human_review_node(state: WorkflowState) -> Dict[str, Any]:
    preview = state.get("generated_test", "")

    if len(preview) > 1200:
        preview = preview[:1200] + "\n...<truncated>"

    # UI Layer (only display data)
    hitl_payload = {
        "title": "Human approval required before final output",
        "instructions": [
            "Reply with:",
            "{'decision': 'approve'}",
            "{'decision': 'revise', 'feedback': '...'}",
        ],
        "preview": preview,
        "action_requests": [
            {"description": "Review generated test and decide next step"}
        ],
    }

    display(HTML(f"<pre style='border:1px dashed red;padding:10px'>{hitl_payload}</pre>"))

    # Execution Layer
    if interrupt is None:
        decision = _next_hitl_decision(allow_input=True)
    else:
        decision = interrupt(hitl_payload)

    approved = False
    feedback = ""

    if isinstance(decision, dict):
        mode = str(decision.get("decision", "")).lower().strip()
        approved = mode in {"approve", "approved", "yes", "y", "true"}
        feedback = str(decision.get("feedback", "")).strip()

    elif isinstance(decision, bool):
        approved = decision

    elif decision is None:
        approved = False

    else:
        text = str(decision).strip()
        approved = text.lower() in {"approve", "approved", "yes", "y", "true"}
        if not approved:
            feedback = text

    updates: Dict[str, Any] = {
        "approved": approved,
        "hitl_payload": hitl_payload,
    }

    if feedback:
        updates["hitl_feedback"] = [feedback]

        try:
            new_ids = persist_user_feedback_memory(
                user_id=state.get("user_id", "anonymous"),
                framework=state.get("framework", ""),
                session_id=state.get("session_id", ""),
                feedback=feedback,
            )

            if new_ids:
                updates["user_memory_ids"] = (
                    state.get("user_memory_ids", []) or []
                ) + new_ids

        except Exception:
            pass

    return updates


def route_after_human_review(state: WorkflowState) -> str:
    return "finalize" if state.get("approved", False) else "generator"


def finalize_node(state: WorkflowState) -> Dict[str, Any]:
    final_payload = {
        "user_id": state["user_id"],
        "session_id": state["session_id"],
        "story_source": state["story_input"],
        "story_text": state["story_text"],
        "framework": state["framework"],
        "acceptance_criteria": json.loads(state["acceptance_criteria_json"]),
        "locators": json.loads(state["locators_json"]),
        "generated_test": state["generated_test"],
        "human_feedback_history": state.get("hitl_feedback", []),
        "retrieval_example_ids": state.get("retrieval_example_ids", []),
        "user_memory_ids": state.get("user_memory_ids", []),
    }

    return {"final_output": json.dumps(final_payload, indent=2)}


print("Graph nodes ready: human_review_node, finalize_node")

In [ ]:
def persist_run_memory_node(state: WorkflowState) -> Dict[str, Any]:
    run_summary = ""
    try:
        payload = json.loads(state.get("final_output", "{}"))
        story_text = payload.get("story_text", "")
        framework = payload.get("framework", "")
        acceptance = payload.get("acceptance_criteria", {})
        criteria = acceptance.get("criteria", []) if isinstance(acceptance, dict) else acceptance
        criteria_count = len(criteria) if isinstance(criteria, list) else 0
        retrieved = payload.get("retrieval_example_ids", [])
        generated_preview = _truncate_text(payload.get("generated_test", ""), max_chars=600)

        run_summary = "\n".join(
            [
                f"Story: {story_text}",
                f"Framework: {framework}",
                f"Criteria count: {criteria_count}",
                f"Retrieved examples: {', '.join(retrieved) if retrieved else 'none'}",
                "Generated test preview:",
                generated_preview,
            ]
        ).strip()

        _ensure_run_memory_collection()
        embedder = OpenAIEmbeddings(
            model=EMBEDDING_MODEL,
            api_key=openai_api_key.get_secret_value(),
        )
        embedding = embedder.embed_query(run_summary)
        story_hash = _hash_text(f"{story_text}-{framework}")
        timestamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
        run_id = f"run-{story_hash}-{timestamp}"
        CHROMA_RUN_MEMORY.add(
            ids=[run_id],
            documents=[run_summary],
            embeddings=[embedding],
            metadatas=[
                {
                    "framework": framework,
                    "story_hash": story_hash,
                    "timestamp": timestamp,
                }
            ],
        )
    except Exception:
        if not run_summary:
            run_summary = "Run memory persistence skipped."

    return {"run_summary": run_summary}


print("Graph node ready: persist_run_memory_node")

In [ ]:
# Persistent checkpointer (SQLite) if available; fallback to in-memory
CHECKPOINT_DB_PATH = Path(os.getcwd()) / "checkpoints" / "qa_checkpoints.sqlite"
CHECKPOINT_DB_PATH.parent.mkdir(parents=True, exist_ok=True)

_CHECKPOINT_CONN = None

if SqliteSaver is not None:
    try:
        _CHECKPOINT_CONN = sqlite3.connect(
            str(CHECKPOINT_DB_PATH),
            check_same_thread=False
        )
        checkpointer = SqliteSaver(_CHECKPOINT_CONN)  # type: ignore
        print(f"Using SqliteSaver checkpointer at: {CHECKPOINT_DB_PATH}")
    except Exception as e:
        print(f"[Checkpoint fallback] Failed to init SqliteSaver: {e}")
        checkpointer = MemorySaver()
else:
    print("SqliteSaver unavailable; using MemorySaver (in-memory checkpoints).")
    checkpointer = MemorySaver()

builder = StateGraph(WorkflowState)

# Nodes
builder.add_node("parse_request", parse_request_node)
builder.add_node("retrieve_run_memory", retrieve_run_memory_node)
builder.add_node("analyst", analyst_node)
builder.add_node("retrieve_user_memory", retrieve_user_memory_node)
builder.add_node("retrieve_context", retrieve_context_node)
builder.add_node("precompute_locators", precompute_locators_node)
builder.add_node("prep_join", prep_join_node)
builder.add_node("generator", generator_node)
builder.add_node("validator", validator_node)
builder.add_node("human_review", human_review_node)
builder.add_node("finalize", finalize_node)
builder.add_node("persist_run_memory", persist_run_memory_node)

print("Graph builder ready (nodes registered).")

In [ ]:
# Flow
builder.set_entry_point("parse_request")

builder.add_edge("parse_request", "retrieve_run_memory")
builder.add_edge("retrieve_run_memory", "analyst")

# Fan-out
builder.add_edge("analyst", "retrieve_user_memory")
builder.add_edge("analyst", "retrieve_context")
builder.add_edge("analyst", "precompute_locators")

# Fan-in
builder.add_edge("retrieve_user_memory", "prep_join")
builder.add_edge("retrieve_context", "prep_join")
builder.add_edge("precompute_locators", "prep_join")

builder.add_edge("prep_join", "generator")
builder.add_edge("generator", "validator")

builder.add_conditional_edges(
    "validator",
    route_after_validation,
    {
        "generator": "generator",
        "human_review": "human_review",
    },
)

builder.add_conditional_edges(
    "human_review",
    route_after_human_review,
    {
        "finalize": "finalize",
        "generator": "generator",
    },
)

builder.add_edge("finalize", "persist_run_memory")
builder.add_edge("persist_run_memory", END)

app = builder.compile(checkpointer=checkpointer)

print("Workflow graph compiled.")
display_graph(app)

In [ ]:
def execute_workflow(user_request: str) -> Dict[str, Any]:
    """
    Core function required by the assignment.
    Accepts one string parameter and runs the workflow end-to-end.

    Supports:
    - Normal execution with optional auto-resume from queued HITL decisions.
    - Resume execution via payload: {"resume": {"thread_id": "...", "decision": {...}}}
    """

    try:
        payload = json.loads(user_request)
    except Exception:
        payload = {"story_input": user_request}

    allow_input = bool(payload.get("allow_input", False))

    resume_payload = payload.get("resume")

    if isinstance(resume_payload, dict) and resume_payload.get("thread_id"):
        thread_id = str(resume_payload.get("thread_id"))
        decision = resume_payload.get("decision")

        config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 50}

        if Command is None:
            raise RuntimeError(
                "Resume requested but Command is unavailable in this LangGraph version."
            )

        try:
            result = app.invoke(Command(resume=decision), config=config)
            status = "complete"
        except Exception as ex:
            if GraphInterrupt is not None and isinstance(ex, GraphInterrupt):
                status = "needs_review"
                hitl_payload = getattr(ex, "value", None)
                return {
                    "status": status,
                    "thread_id": thread_id,
                    "hitl_payload": hitl_payload,
                }
            raise

        return {
            "status": status,
            "thread_id": thread_id,
            "user_id": result.get("user_id", ""),
            "session_id": result.get("session_id", ""),
            "story_input": result.get("story_input", ""),
            "framework": result.get("framework", ""),
            "analysis": result.get("analysis", ""),
            "validation_errors": result.get("validation_errors", []),
            "validation_summary": result.get("validation_summary", ""),
            "retrieval_example_ids": result.get("retrieval_example_ids", []),
            "generated_test": result.get("generated_test", ""),
            "final_output": result.get("final_output", "{}"),
        }

    # Normal execution
    payload.setdefault("user_id", "anonymous")
    payload.setdefault("session_id", uuid.uuid4().hex[:8])

    normalized_request = json.dumps(payload)
    parsed = parse_user_request_text(normalized_request)
    safe_user_id = _slugify(parsed["user_id"])
    thread_id = f"qa_{safe_user_id}_{parsed['session_id']}"

    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 50}

    next_input: Any = {"user_request": normalized_request, "hitl_feedback": []}
    max_interrupt_resumes = 20

    for _ in range(max_interrupt_resumes):
        try:
            result = app.invoke(input=next_input, config=config)
            status = "complete"
            break
        except Exception as ex:
            if GraphInterrupt is None or not isinstance(ex, GraphInterrupt):
                raise

            hitl_payload = getattr(ex, "value", None)

            # Auto-resume for demo notebooks using queued decisions
            decision = _next_hitl_decision(allow_input=allow_input)

            if decision is None:
                return {
                    "status": "needs_review",
                    "thread_id": thread_id,
                    "user_id": parsed["user_id"],
                    "session_id": parsed["session_id"],
                    "hitl_payload": hitl_payload,
                }

            if Command is None:
                raise RuntimeError(
                    "Interrupt received but Command is unavailable; cannot resume."
                )

            next_input = Command(resume=decision)

    else:
        raise RuntimeError("Exceeded max interrupt resumes; possible infinite loop.")

    return {
        "status": status,
        "thread_id": thread_id,
        "user_id": parsed["user_id"],
        "session_id": parsed["session_id"],
        "story_input": result.get("story_input", ""),
        "framework": result.get("framework", ""),
        "analysis": result.get("analysis", ""),
        "validation_errors": result.get("validation_errors", []),
        "validation_summary": result.get("validation_summary", ""),
        "retrieval_example_ids": result.get("retrieval_example_ids", []),
        "generated_test": result.get("generated_test", ""),
        "final_output": result.get("final_output", "{}"),
    }


print("execute_workflow(user_request) is ready.")

## Examples & Test Cases

Quick run examples and assignment test cases (includes simulated HITL responses).

In [ ]:
# Test case run list (shared across the following cells)
run_summaries = []
print("Initialized run_summaries.")

In [ ]:
# Test Case #1: Jira login story -> Cypress (approve) [quick run example]
print("=" * 90)
print("Test Case #1: Jira login story -> Cypress (approve)")
HITL_TEST_DECISIONS.clear()
enqueue_hitl_responses({"decision": "approve"})

tc1_request = {
    "user_id": "qa_lead",
    "session_id": "login-001",
    "story_input": "QA-101",
    "framework": "cypress-ts",
    "html_dom": "login_page",
}
tc1_output = execute_workflow(json.dumps(tc1_request))

print("Status:", tc1_output.get("status", ""))
print("Validation:", tc1_output.get("validation_summary", ""))

print("Thread:", tc1_output["thread_id"])
print("Framework:", tc1_output["framework"])
print("Retrieved examples:", tc1_output.get("retrieval_example_ids", []))
print("Generated test preview:\n")
print(tc1_output["generated_test"][:900])

run_summaries.append(
    {
        "case": "Jira login story -> Cypress (approve)",
        "thread_id": tc1_output["thread_id"],
        "framework": tc1_output["framework"],
        "status": tc1_output.get("status", ""),
    }
)

In [ ]:
# Test Case #2: Jira cart story -> Playwright (revise then approve)
print("=" * 90)
print("Test Case #2: Jira cart story -> Playwright (revise then approve)")
HITL_TEST_DECISIONS.clear()
enqueue_hitl_responses(
    {
        "decision": "revise",
        "feedback": "Add an explicit assertion for post-action feedback and keep selector strategy robust.",
    }
)
enqueue_hitl_responses({"decision": "approve"})

tc2_request = {
    "user_id": "qa_analyst",
    "session_id": "cart-001",
    "story_input": "QA-102",
    "framework": "playwright-ts",
    "html_dom": "cart_page",
}
tc2_output = execute_workflow(json.dumps(tc2_request))

print("Status:", tc2_output.get("status", ""))
print("Validation:", tc2_output.get("validation_summary", ""))

print("Thread:", tc2_output["thread_id"])
print("Framework:", tc2_output["framework"])
print("Retrieved examples:", tc2_output.get("retrieval_example_ids", []))
print("Generated test snippet:")
print(tc2_output["generated_test"][:450], "...\n")

run_summaries.append(
    {
        "case": "Jira cart story -> Playwright (revise then approve)",
        "thread_id": tc2_output["thread_id"],
        "framework": tc2_output["framework"],
        "status": tc2_output.get("status", ""),
    }
)

In [ ]:
# Test Case #3: Raw story text (reset password) -> Cypress (approve)
# Note: provide a matching DOM snippet for reset-password flow (instead of login DOM).
print("=" * 90)
print("Test Case #3: Raw story text (reset password) -> Cypress (approve)")
HITL_TEST_DECISIONS.clear()
enqueue_hitl_responses({"decision": "approve"})

tc3_request = {
    "user_id": "qa_operator",
    "session_id": "reset-001",
    "story_input": "As a customer, I want to reset my password so that I can recover access quickly.",
    "framework": "cypress-ts",
    "html_dom": """
<main data-testid='reset-password-page'>
  <form data-testid='reset-form'>
    <input data-testid='email-input' name='email' type='email' />
    <button data-testid='reset-button' type='submit'>Reset Password</button>
  </form>
  <div data-testid='confirmation' role='status' hidden>Check your email</div>
</main>
""",
}
tc3_output = execute_workflow(json.dumps(tc3_request))

print("Status:", tc3_output.get("status", ""))
print("Validation:", tc3_output.get("validation_summary", ""))

print("Thread:", tc3_output["thread_id"])
print("Framework:", tc3_output["framework"])
print("Retrieved examples:", tc3_output.get("retrieval_example_ids", []))
print("Generated test snippet:")
print(tc3_output["generated_test"][:450], "...\n")

run_summaries.append(
    {
        "case": "Raw story text (reset password) -> Cypress (approve)",
        "thread_id": tc3_output["thread_id"],
        "framework": tc3_output["framework"],
        "status": tc3_output.get("status", ""),
    }
)

In [ ]:
# Test Case #4: Unknown Jira key fallback -> Playwright (approve)
print("=" * 90)
print("Test Case #4: Unknown Jira key fallback -> Playwright (approve)")
HITL_TEST_DECISIONS.clear()
enqueue_hitl_responses({"decision": "approve"})

tc4_request = {
    "user_id": "qa_lead",
    "session_id": "unknown-001",
    "story_input": "QA-999",
    "framework": "playwright-ts",
    "html_dom": "<main><button aria-label='Submit'>Submit</button></main>",
}
tc4_output = execute_workflow(json.dumps(tc4_request))

print("Status:", tc4_output.get("status", ""))
print("Validation:", tc4_output.get("validation_summary", ""))

print("Thread:", tc4_output["thread_id"])
print("Framework:", tc4_output["framework"])
print("Retrieved examples:", tc4_output.get("retrieval_example_ids", []))
print("Generated test snippet:")
print(tc4_output["generated_test"][:450], "...\n")

run_summaries.append(
    {
        "case": "Unknown Jira key fallback -> Playwright (approve)",
        "thread_id": tc4_output["thread_id"],
        "framework": tc4_output["framework"],
        "status": tc4_output.get("status", ""),
    }
)

In [ ]:
# Test Case #5: Admin deactivation story -> Cypress (approve)
print("=" * 90)
print("Test Case #5: Admin deactivation story -> Cypress (approve)")
HITL_TEST_DECISIONS.clear()
enqueue_hitl_responses({"decision": "approve"})

tc5_request = {
    "user_id": "qa_admin",
    "session_id": "admin-001",
    "story_input": "QA-103",
    "framework": "cypress-ts",
    "html_dom": "admin_page",
}
tc5_output = execute_workflow(json.dumps(tc5_request))

print("Status:", tc5_output.get("status", ""))
print("Validation:", tc5_output.get("validation_summary", ""))

print("Thread:", tc5_output["thread_id"])
print("Framework:", tc5_output["framework"])
print("Retrieved examples:", tc5_output.get("retrieval_example_ids", []))
print("Generated test snippet:")
print(tc5_output["generated_test"][:450], "...\n")

run_summaries.append(
    {
        "case": "Admin deactivation story -> Cypress (approve)",
        "thread_id": tc5_output["thread_id"],
        "framework": tc5_output["framework"],
        "status": tc5_output.get("status", ""),
    }
)

In [ ]:
# Test Case #6: Persistence preferences -> Cypress (revise then approve)
print("=" * 90)
print("Test Case #6: Persistence preferences -> Cypress (revise then approve)")
HITL_TEST_DECISIONS.clear()
enqueue_hitl_responses(
    {
        "decision": "revise",
        "feedback": "Use data-testid selectors and add an explicit assertion that preferences persist after reload.",
    }
)
enqueue_hitl_responses({"decision": "approve"})

tc6_request = {
    "user_id": "qa_tester",
    "session_id": "revise-002",
    "story_input": "As a user, I want to save my preferences so they persist across sessions.",
    "framework": "cypress-ts",
    "html_dom": """
<main data-testid='preferences-page'>
  <form data-testid='preferences-form'>
    <label>
      <input data-testid='newsletter-toggle' type='checkbox' /> Newsletter
    </label>
    <button data-testid='save-button' type='button'>Save</button>
  </form>
  <div data-testid='toast' hidden>Saved</div>
</main>
""",
}
tc6_output = execute_workflow(json.dumps(tc6_request))

print("Status:", tc6_output.get("status", ""))
print("Validation:", tc6_output.get("validation_summary", ""))

print("Thread:", tc6_output["thread_id"])
print("Framework:", tc6_output["framework"])
print("Retrieved examples:", tc6_output.get("retrieval_example_ids", []))
print("Generated test snippet:")
print(tc6_output["generated_test"][:450], "...\n")

run_summaries.append(
    {
        "case": "Persistence preferences -> Cypress (revise then approve)",
        "thread_id": tc6_output["thread_id"],
        "framework": tc6_output["framework"],
        "status": tc6_output.get("status", ""),
    }
)

In [ ]:
# Completed test cases summary
print("=" * 90)
print("Completed", len(run_summaries), "test cases.")
run_summaries